In [1]:
from neo4j import GraphDatabase
import networkx as nx
import pickle
import pandas as pd
import re
from tqdm import tqdm
import matplotlib.pyplot as plt
import math
import ast
import igraph as ig
import leidenalg
import pandas as pd

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
print(f'base_dir is {base_dir}')
sys.path.append(os.path.dirname(base_dir))
from util.cmpa import (compute_IF_score, 
                       run_cmpa, 
                       extract_subgraphs_from_neo4j, 
                       convert_graphdict_to_nx,
                       subgraph_to_rules)
                    

base_dir is /Users/yuxiaoxuan/master_thesis/KGRules/notebooks


In [2]:
# Connection to Neo4j
driver_ad = GraphDatabase.driver("bolt://localhost:7690", auth=('neo4j','test1237'))
driver_health = GraphDatabase.driver("bolt://localhost:7688", auth=('neo4j','test12345'))

query_BP = """
        MATCH p=(s)-[]->(o) 
        UNWIND nodes(p) AS n
        UNWIND relationships(p) AS r
        WITH collect(DISTINCT n) AS nodes, collect(DISTINCT r) AS rels
        RETURN nodes, rels
        """
BP_dict = extract_subgraphs_from_neo4j(driver=driver_ad, 
                                            query=query_BP, 
                                            ad=True)
bp_G = convert_graphdict_to_nx(graph_dict=BP_dict, ad=True)
# remove nodes that are not Proteins or BiologicalProcess
# remove nodes that are not protein or biological process
node_to_remove = [n for n, attr in bp_G.nodes(data=True) if attr.get('labels') not in ['Protein', 'BiologicalProcess']]
len(node_to_remove)
bp_G.remove_nodes_from(node_to_remove)
print(f"nodes:{bp_G.number_of_nodes()}, edges:{bp_G.number_of_edges()}")

Added 2978 nodes to subGraph
Added 5075 edges to subGraph
nodes:1350, edges:1290


In [30]:
bp_health_dict = extract_subgraphs_from_neo4j(driver=driver_health, 
                                            query=query_BP, 
                                            ad=False)
bp_health = convert_graphdict_to_nx(graph_dict=bp_health_dict, ad=False)
# remove nodes that are not Proteins or BiologicalProcess

Added 4161 nodes to subGraph
Added 7406 edges to subGraph


In [31]:
for node, attr in bp_health.nodes(data=True):
    bp_nodes = 0
    pr_nodes = 0
    #print(node)
    if node.startswith('bp'):
        bp_nodes += 1
        label = attr['labels']
        new_label = ''.join([word.capitalize() for word in label.split('_')])
        #print(new_label)
        attr['labels'] = new_label
        print(attr)
    if node.startswith('p('):
        pr_nodes += 1
        #print(attr)

{'labels': 'BiologicalProcess', 'protein': '', 'properties': {'out_regulates': '["#814:0", "#815:0", "#812:4", "#815:5"]', 'out_increases': '["#697:0", "#698:0", "#691:1", "#696:1", "#697:1", "#701:1", "#706:1", "#693:2", "#695:2"]', 'bel': 'bp(GOBP:"synaptic signaling")', 'in_regulates': '["#816:5", "#819:5", "#824:5", "#823:14", "#824:15", "#813:17", "#819:17", "#814:18", "#820:18", "#812:20", "#822:26", "#817:33"]', 'type': 'Biological_process', 'involved_genes': '[]', 'involved_other': '["synaptic signaling"]', 'in_increases': '["#705:38", "#697:40", "#697:78", "#701:78"]', 'in_decreases': '["#738:0", "#741:0", "#742:0", "#745:0", "#748:0", "#742:4", "#746:30", "#736:35"]', 'name': 'synaptic signaling', 'namespace': 'GOBP', 'orientdb_rid': '#96:0', 'in_association': '["#870:2"]'}}
{'labels': 'BiologicalProcess', 'protein': '', 'properties': {'out_regulates': '["#819:45"]', 'involved_genes': '[]', 'involved_other': '["lysosomal protein catabolic process"]', 'in_increases': '["#695:8

In [32]:
node_to_remove = [n for n, attr in bp_health.nodes(data=True) if attr.get('labels') not in ['Protein', 'BiologicalProcess']]
len(node_to_remove)
bp_health.remove_nodes_from(node_to_remove)
print(f"BP+Protein Graph nodes:{bp_health.number_of_nodes()}, edges:{bp_health.number_of_edges()}")

BP+Protein Graph nodes:1313, edges:1696


In [33]:
def extract_bp_subgraphs_dedup(G, hop=2, min_size=5):
    bp_nodes = [n for n, d in G.nodes(data=True) if d.get("labels") == "BiologicalProcess"]
    print(bp_nodes)
    
    subgraphs = {}
    protein_coverage = {}  # track which BPs cover each protein
    
    for bp in bp_nodes:
        neighbors = nx.single_source_shortest_path_length(G, bp, cutoff=hop)
        sg = G.subgraph(list(neighbors.keys())).copy()
        
        if sg.number_of_nodes() < min_size:
            continue
        
        subgraphs[bp] = sg
        
        # Track protein membership
        for node in sg.nodes():
            if G.nodes[node].get("labels") == "Protein":
                protein_coverage.setdefault(node, []).append(bp)
    # Report bp-subgrphs stats
    subgraph_num_nodes = [subg.number_of_nodes() for subg in subgraphs.values()]
    print(f"Number of subgraphs: {len(subgraphs)}")
    print(f"Subgraph node count: min={min(subgraph_num_nodes)}, max={max(subgraph_num_nodes)}")
    # Report overlap stats
    overlap = {p: bps for p, bps in protein_coverage.items() if len(bps) > 1}
    print(f"\nProteins appearing in multiple BP subgraphs: {len(overlap)}")
    print(f"Max overlap: {max(len(v) for v in protein_coverage.values())} BPs")
    
    return subgraphs, protein_coverage

In [34]:
subgraph_health, proteins_health = extract_bp_subgraphs_dedup(bp_health)

['bp(GOBP:"synaptic signaling")', 'bp(GOBP:"lysosomal protein catabolic process")', 'bp(GOBP:"neuron projection development")', 'bp(GOBP:"neuron projection arborization")', 'bp(GOBP:"dendrite development")', 'bp(GOBP:"protein stabilization")', 'bp(GOBP:"fatty acid beta-oxidation")', 'bp(GOBP:"D-glucose transmembrane transport")', 'bp(GOBP:"cholesterol efflux")', 'bp(MESHPP:"Neuroprotection")', 'bp(GOBP:"amyloid-beta clearance")', 'bp(FIXME:"immature neuron migration")', 'bp(GOBP:"dendritic spine development")', 'bp(GOBP:"long-term synaptic potentiation")', 'bp(GOBP:"cell death")', 'bp(MESHPP:"Transcriptional Activation")', 'bp(GOBP:"apoptotic process")', 'bp(GOBP:"memory")', 'bp(GOBP:"calcium-mediated signaling")', 'bp(MESHPP:"Neuronal Plasticity")', 'bp(GOBP:"learning")', 'bp(GOBP:"calcium ion homeostasis")', 'bp(GOBP:"monocyte chemotaxis")', 'bp(MESHPP:"Cell Survival")', 'bp(FIXME:"alpha-Secretase cleavage")', 'bp(GOBP:"membrane hyperpolarization")', 'bp(GOBP:"actin cytoskeleton orga

In [6]:
subgraphs, protein_coverage = extract_bp_subgraphs_dedup(bp_G)

Number of subgraphs: 42
Subgraph node count: min=5, max=144

Proteins appearing in multiple BP subgraphs: 109
Max overlap: 23 BPs


In [7]:
print(len(protein_coverage))
protein_coverage

163


{'p(HGNC:"NOS2")': ['bp(GO:"inflammatory response")',
  'bp(GO:"macrophage activation")',
  'bp(GO:"regulation of cAMP-dependent protein kinase activity")',
  'bp(GO:"viral process")',
  'bp(GO:"myeloid dendritic cell activation")'],
 'p(HGNC:"RELA")': ['bp(GO:"inflammatory response")',
  'bp(GO:"macrophage activation")',
  'bp(GO:"myeloid dendritic cell activation")'],
 'p(HGNC:"BACE1")': ['bp(GO:"inflammatory response")',
  'bp(GO:"ERK1 and ERK2 cascade")',
  'bp(GO:"response to oxidative stress")',
  'bp(GO:"maintenance of permeability of blood-brain barrier")',
  'bp(GO:"chemical synaptic transmission")',
  'bp(GO:"endocannabinoid signaling pathway")',
  'bp(GO:"macrophage activation")',
  'bp(PTS:"p53 signaling pathway")'],
 'p(HGNC:"IDE")': ['bp(GO:"inflammatory response")', 'bp(GO:"viral process")'],
 'p(HGNC:"TLR4")': ['bp(GO:"inflammatory response")',
  'bp(GO:"macrophage activation")',
  'bp(GO:"myeloid dendritic cell activation")'],
 'p(HGNC:"GRB2")': ['bp(GO:"response to en

In [13]:
expression_path:str="../Anyburl/data/adni_gene_cleaned.csv"
exp_df = pd.read_csv(expression_path, index_col=0).T
df_patient_cmpa = pd.DataFrame(index=exp_df.index, columns = list(subgraphs.keys()))
    
for i in tqdm(range(len(exp_df)), desc="Computing Subgraph CMPA Score For Patients"):
    patient_data = exp_df.iloc[i].to_dict()

    df_subgraphs = {}
    subgraph_scores = []
    for subg_name, subG in subgraphs.items():
        #plot_subgraph(subG, True, True, subg_name)
        print('='*60)
        print(f"Run CMPA on {subg_name}")
        df_subG = run_cmpa(patient_data, subG)
        df_subgraphs[subg_name] = df_subG
        
        subgraph_scores.append(float(df_subG[df_subG['hub'] == subg_name]['score']))
    df_patient_cmpa.iloc[i] = subgraph_scores
    break
    

Computing Subgraph CMPA Score For Patients:   0%|          | 0/744 [00:00<?, ?it/s]

Run CMPA on bp(GO:"inflammatory response")
Graph is a DAG. Running standard topological CMPA.


Iterative convergence:   4%|▍         | 1/26 [00:00<00:00, 7869.24it/s]
/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_81331/317742454.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  subgraph_scores.append(float(df_subG[df_subG['hub'] == subg_name]['score']))


Run CMPA on bp(GO:"response to endoplasmic reticulum stress")


Iterative convergence: 100%|██████████| 38/38 [00:00<00:00, 5310.31it/s]


...

Iterative convergence:   9%|▉         | 1/11 [00:00<00:00, 4848.91it/s]


Run CMPA on bp(GO:"myeloid dendritic cell activation")
Graph is a DAG. Running standard topological CMPA.


Iterative convergence:  10%|█         | 1/10 [00:00<00:00, 9177.91it/s]


Run CMPA on bp(PTS:"p53 signaling pathway")
Graph is a DAG. Running standard topological CMPA.


Computing Subgraph CMPA Score For Patients:   0%|          | 0/744 [00:00<?, ?it/s]


In [12]:
df_subG[df_subG['hub'] == subg_name]['score']

1    1.0
Name: score, dtype: float64

In [16]:
df_patient+cmpa[df_patient_cmpa[column].nunique()>1 for column in df_patient_cmpa.columns]

SyntaxError: invalid syntax (1978855686.py, line 1)